# 04 - Publish example nanopublications

Builds the example set from the published templates and publishes it. Part A is
a worked thread, part B one instance of each template. Every instance is marked
`npx:ExampleNanopub` and stamps `nt:wasCreatedFromTemplate`. The content is
deliberately silly so nobody mistakes it for real findings.

`TEST = True` targets the test server. Production (`TEST = False`) is
permanent: a nanopub can only be retracted or superseded.

In [ ]:
import sys, subprocess
try:
    import nanopub, rdflib
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "nanopub", "rdflib"])
import na_nanopub as na


## Configuration

In [ ]:
TEST = True   # True = test server. False = PRODUCTION (permanent).

DISCOURSE  = "https://w3id.org/np/RAFpID-NunXGUAZebXPfhD9efjvfpZqB5gDBrgP2Lp5N8"
ANNOTATION = "https://w3id.org/np/RATxyPikKUvLwqijI4T_fvaHE758DsxgXhOq0m6KiLlEI"
EVIDENCE   = "https://w3id.org/np/RAdGQVeqqUoTNJe52OSMhY8EX1MkFA1LI4oQh-NAgrYV8"
QUESTION   = "https://w3id.org/np/RAOp9i7XUKKJhT4qIvtCA6LpXj9Hq7qkhsZ1MrLP7UfHE"

def stamp(template):
    return na.from_template(template) + "\n" + na.EXAMPLE

print("Target:", "TEST server" if TEST else "PRODUCTION network (permanent!)")

## Part A: a worked thread

Signed in publication order; each contribution references its target's trusty
URI.

In [ ]:
root = na.make(
    """  sub:claim a schema:Statement ;
    rdf:value             "Cats knock objects off tables primarily to test local gravity." ;
    cito:citesAsEvidence  <https://example.org/studies/cat-gravity> .""",
    attributed_to="orcid:0000-0001-alvarez-b", nanopub_type="schema:Statement",
    introduces="sub:claim", extra_pubinfo=stamp(DISCOURSE), test=TEST, name="04-root")
claim = root.source_uri + "/claim"

support = na.make(
    f"""  sub:reason a schema:Statement ;
    rdf:value "Home video evidence shows consistent downward acceleration in all recorded incidents." ;
    cito:supports <{claim}> .""",
    attributed_to="orcid:0000-0002-wang-p", nanopub_type="cito:supports",
    introduces="sub:reason", extra_pubinfo=stamp(DISCOURSE), test=TEST, name="04-support")

caveat = na.make(
    f"""  sub:caveat a schema:Statement ;
    rdf:value "Results may be confounded by cats also enjoying the sound of things breaking." ;
    cito:qualifies <{claim}> .""",
    attributed_to="<https://example.org/agents/medai-bot>", nanopub_type="cito:qualifies",
    introduces="sub:caveat", extra_pubinfo=stamp(DISCOURSE), test=TEST, name="04-caveat")

counter = na.make(
    f"""  sub:counterclaim a schema:Statement ;
    rdf:value "The behavior is better explained by a desire for human attention." ;
    cito:disputes <{claim}> .""",
    attributed_to="orcid:0000-0003-moller-m", nanopub_type="cito:disputes",
    introduces="sub:counterclaim", extra_pubinfo=stamp(DISCOURSE), test=TEST, name="04-counter")
counterclaim = counter.source_uri + "/counterclaim"

rebuttal = na.make(
    f"""  sub:rebuttal a schema:Statement ;
    rdf:value             "Attention-seeking does not explain the incidents recorded at 3am in empty rooms." ;
    cito:citesAsEvidence  <https://example.org/studies/cat-gravity> ;
    as:inReplyTo          <{counterclaim}> ;
    cito:disputes         <{counterclaim}> .""",
    attributed_to="<https://example.org/agents/criticai-bot>", nanopub_type="cito:disputes",
    introduces="sub:rebuttal", extra_pubinfo=stamp(DISCOURSE), test=TEST, name="04-rebuttal")

question = na.make(
    f"""  sub:question a schema:Question ;
    rdf:value     "Has anyone repeated this with round tables?" ;
    as:inReplyTo  <{claim}> .""",
    attributed_to="orcid:0000-0002-perez-l", nanopub_type="schema:Question",
    introduces="sub:question", extra_pubinfo=stamp(QUESTION), test=TEST, name="04-thread-question")

thread = [root, support, caveat, counter, rebuttal, question]
for np in thread:
    na.show(np)

## Part B: one of each template

In [ ]:
disc = na.make(
    """  sub:claim a schema:Statement ;
    rdf:value "Sandwiches taste better when cut diagonally." .""",
    attributed_to="orcid:0000-0001-alvarez-b", nanopub_type="schema:Statement",
    introduces="sub:claim", extra_pubinfo=stamp(DISCOURSE), test=TEST, name="04-one-discourse")
disc_claim = disc.source_uri + "/claim"

ques = na.make(
    f"""  sub:question a schema:Question ;
    rdf:value      "Does this hold for toast?" ;
    cito:disputes  <{disc_claim}> .""",
    attributed_to="orcid:0000-0002-perez-l", nanopub_type="schema:Question",
    introduces="sub:question", extra_pubinfo=stamp(QUESTION), test=TEST, name="04-one-question")

evid = na.make(
    f"""  sub:finding a schema:Statement, sio:SIO_001394 ;
    rdf:value     "In a sample of 1,200 lunches, diagonally cut sandwiches were rated 18% tastier." ;
    cito:supports <{disc_claim}> .""",
    attributed_to="orcid:0000-0002-wang-p", nanopub_type="cito:supports",
    introduces="sub:finding", extra_pubinfo=stamp(EVIDENCE), test=TEST, name="04-one-evidence")

anno = na.make(
    f"""  sub:statement a schema:Statement ;
    rdf:value     "This overstates what the lunch data supports." ;
    cito:disputes <{disc_claim}> .
  sub:annotation a oa:Annotation ;
    oa:hasBody     sub:statement ;
    oa:hasTarget   sub:specificResource ;
    oa:motivatedBy oa:assessing .
  sub:specificResource a oa:SpecificResource ;
    oa:hasSource   <https://example.org/sandwich-report> ;
    oa:hasSelector sub:selector .
  sub:selector a oa:TextQuoteSelector ;
    oa:exact  "diagonal cutting is unambiguously superior" ;
    oa:prefix "we conclude that " ;
    oa:suffix " across all breads" .""",
    attributed_to="<https://example.org/agents/reviewer-bot>", nanopub_type="cito:disputes",
    introduces="sub:statement", extra_pubinfo=stamp(ANNOTATION), test=TEST, name="04-one-annotation")

one_of_each = [disc, ques, evid, anno]
for np in one_of_each:
    na.show(np)

## Publish

In [ ]:
examples = {
    "root": root, "support": support, "caveat": caveat, "counter": counter,
    "rebuttal": rebuttal, "thread-question": question,
    "one-discourse": disc, "one-question": ques, "one-evidence": evid, "one-annotation": anno,
}

published = {}
for label, np in examples.items():
    uri = na.publish(np)
    published[label] = uri
    print(f"published  {label:16s}  {uri}")


## Verify: the discourse query retrieves the thread

In [ ]:
ds = na.load(*thread)
q = na.QP + f"""
SELECT ?relation ?contribution ?value WHERE {{ GRAPH ?g {{
  ?contribution ?relation <{claim}> ; rdf:value ?value .
  FILTER(?relation IN (cito:supports, cito:disputes, cito:extends,
                       cito:agreesWith, cito:qualifies, as:inReplyTo))
}} }} ORDER BY ?relation"""

buckets = {}
for r in ds.query(q):
    buckets.setdefault(na.localname(r.relation), []).append(str(r.value))

print(f"Claim: {claim}\n")
for rel, items in buckets.items():
    print(f"[{rel}]  ({len(items)})")
    for v in items:
        print(f"   - {v}")


## Published URIs

In [ ]:
for label, uri in published.items():
    print(f"{label:16s}  {uri}")
